# Delta vs Iceberg vs Hudi — The Format Wars

This notebook is written from the perspective of a Senior Data Engineer comparing modern lakehouse table formats for a telemetry-heavy enterprise data platform.

## Mental model

Think of the three formats like this:

- **Delta Lake**: the smoothest path inside **Databricks-first** Spark + SQL ecosystems.
- **Apache Iceberg**: the strongest choice for **multi-engine open architectures** where Spark, Trino, Flink, Snowflake, and query engines all need to agree on table state.
- **Apache Hudi**: the specialist for **high-frequency upserts and CDC-heavy pipelines**.

## Dataset context used throughout

- PostgreSQL database: `de_telemetry` on `localhost:5432`
- `endpoints`: 10,000 rows
- `metrics`: 500,000 rows
- `alerts`: 25,000 rows
- Citi-style narrative: 6,000+ API endpoints monitored for latency, error rate, throughput; alerts escalate through severity tiers

## Tech stack context

- Kafka: `localhost:9092`
- Spark: `pyspark==3.5.4`, `master=local[*]`
- Airflow: `localhost:8082`
- MLflow: `localhost:5000`
- dbt target: PostgreSQL
- Databricks host: `https://dbc-9f35a83d-b4e7.cloud.databricks.com`
- Databricks Serverless SQL Warehouse: `b6657f31d1e7a179`

The notebook contains:

1. A **live Delta Lake demo** against Databricks when the connector and auth are available
2. A **comparison matrix**
3. Deep dives on **Iceberg** and **Hudi**
4. A **decision recommendation** for the Citi telemetry scenario


In [ ]:

from __future__ import annotations

import os
import math
import json
import time
import textwrap
from typing import Optional, Tuple

import pandas as pd

# Real values from the provided environment context
PG_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!",
}

DATABRICKS_CONFIG = {
    "server_hostname": "dbc-9f35a83d-b4e7.cloud.databricks.com",
    "http_path": "/sql/1.0/warehouses/b6657f31d1e7a179",
    "host_url": "https://dbc-9f35a83d-b4e7.cloud.databricks.com",
    "warehouse_id": "b6657f31d1e7a179",
}

def print_header(title: str) -> None:
    print("\\n" + "=" * 100)
    print(title)
    print("=" * 100)

print_header("Environment configuration loaded")
print("PostgreSQL:", PG_CONFIG)
print("Databricks:", DATABRICKS_CONFIG)
print("\\nNotebook goal: compare Delta Lake vs Apache Iceberg vs Apache Hudi.")


In [ ]:

def load_postgres_sample() -> pd.DataFrame:
    """
    Attempts to read a small telemetry-shaped sample from PostgreSQL.
    Falls back to synthetic data if psycopg2 is unavailable or the DB is not reachable.
    This keeps the notebook executable top-to-bottom without crashing.
    """
    try:
        import psycopg2  # type: ignore
        conn = psycopg2.connect(
            host=PG_CONFIG["host"],
            port=PG_CONFIG["port"],
            dbname=PG_CONFIG["database"],
            user=PG_CONFIG["user"],
            password=PG_CONFIG["password"],
        )
        query = """
        SELECT
            m.endpoint_id,
            e.name AS endpoint_name,
            e.region,
            e.status,
            e.category,
            m.metric_name,
            m.value,
            m.timestamp
        FROM metrics m
        JOIN endpoints e
          ON e.endpoint_id = m.endpoint_id
        ORDER BY m.timestamp DESC
        LIMIT 1000
        """
        df = pd.read_sql_query(query, conn)
        conn.close()
        print("Loaded 1000 rows from PostgreSQL.")
        return df
    except Exception as exc:
        print("PostgreSQL sample load skipped:", exc)
        print("Building deterministic synthetic sample instead.")
        rows = []
        metric_names = ["latency_ms", "error_rate", "throughput_rps"]
        regions = ["us-east-1", "us-west-2", "eu-west-1"]
        statuses = ["healthy", "warning", "critical"]
        categories = ["payments", "cards", "auth", "risk"]
        base_ts = pd.Timestamp("2026-03-01 00:00:00", tz="UTC")
        for i in range(1000):
            endpoint_id = (i % 250) + 1
            metric_name = metric_names[i % len(metric_names)]
            if metric_name == "latency_ms":
                value = 80 + (i % 90)
            elif metric_name == "error_rate":
                value = round(((i % 12) / 100.0), 4)
            else:
                value = 250 + (i % 400)
            rows.append(
                {
                    "endpoint_id": endpoint_id,
                    "endpoint_name": f"api_endpoint_{endpoint_id}",
                    "region": regions[i % len(regions)],
                    "status": statuses[i % len(statuses)],
                    "category": categories[i % len(categories)],
                    "metric_name": metric_name,
                    "value": value,
                    "timestamp": base_ts + pd.Timedelta(minutes=i),
                }
            )
        return pd.DataFrame(rows)

demo_df = load_postgres_sample()
print(demo_df.head(5).to_string(index=False))
print("\\nShape:", demo_df.shape)


In [ ]:

def get_databricks_connection():
    """
    Uses databricks-sql-connector if available and a token is supplied via env var.
    Expected env vars:
      - DATABRICKS_TOKEN
    """
    token = os.getenv("DATABRICKS_TOKEN", "").strip()
    if not token:
        print("DATABRICKS_TOKEN not set. Live Databricks execution will be skipped.")
        return None

    try:
        from databricks import sql  # type: ignore
    except Exception as exc:
        print("databricks-sql-connector import failed. Live Databricks execution will be skipped:", exc)
        return None

    try:
        conn = sql.connect(
            server_hostname=DATABRICKS_CONFIG["server_hostname"],
            http_path=DATABRICKS_CONFIG["http_path"],
            access_token=token,
        )
        print("Connected to Databricks SQL Warehouse.")
        return conn
    except Exception as exc:
        print("Databricks connection failed. Live Databricks execution will be skipped:", exc)
        return None


def databricks_execute(conn, statement: str, fetch: bool = False) -> Optional[pd.DataFrame]:
    with conn.cursor() as cur:
        cur.execute(statement)
        if fetch:
            rows = cur.fetchall()
            columns = [desc[0] for desc in cur.description]
            return pd.DataFrame(rows, columns=columns)
    return None


print_header("Delta Lake live demo")
conn = get_databricks_connection()

if conn is None:
    print(textwrap.dedent(f"""
    Live demo skipped safely.

    To run it for real in your environment:
      1. Ensure databricks-sql-connector is installed
      2. Set env var DATABRICKS_TOKEN
      3. Re-run this cell

    Target object:
      catalog/schema.table = citi_delta.format_comparison
      warehouse          = {DATABRICKS_CONFIG["warehouse_id"]}
      host               = {DATABRICKS_CONFIG["host_url"]}
    """))
else:
    create_schema_sql = "CREATE SCHEMA IF NOT EXISTS citi_delta"
    create_table_sql = """
    CREATE TABLE IF NOT EXISTS citi_delta.format_comparison (
        endpoint_id BIGINT,
        endpoint_name STRING,
        region STRING,
        status STRING,
        category STRING,
        metric_name STRING,
        value DOUBLE,
        ts TIMESTAMP
    ) USING DELTA
    """
    truncate_sql = "DELETE FROM citi_delta.format_comparison WHERE 1 = 1"

    databricks_execute(conn, create_schema_sql)
    databricks_execute(conn, create_table_sql)
    databricks_execute(conn, truncate_sql)

    rows_sql = []
    for row in demo_df.itertuples(index=False):
        endpoint_name = str(row.endpoint_name).replace("'", "''")
        region = str(row.region).replace("'", "''")
        status = str(row.status).replace("'", "''")
        category = str(row.category).replace("'", "''")
        metric_name = str(row.metric_name).replace("'", "''")
        ts = pd.Timestamp(row.timestamp).strftime("%Y-%m-%d %H:%M:%S")
        rows_sql.append(
            f"({int(row.endpoint_id)}, '{endpoint_name}', '{region}', '{status}', '{category}', '{metric_name}', {float(row.value)}, TIMESTAMP '{ts}')"
        )

    insert_sql = "INSERT INTO citi_delta.format_comparison VALUES\n" + ",\n".join(rows_sql)
    databricks_execute(conn, insert_sql)

    count_df = databricks_execute(conn, "SELECT COUNT(*) AS row_count FROM citi_delta.format_comparison", fetch=True)
    print("Inserted row count:")
    print(count_df.to_string(index=False))

    history_df = databricks_execute(conn, "DESCRIBE HISTORY citi_delta.format_comparison", fetch=True)
    print("\\nDESCRIBE HISTORY output:")
    print(history_df.head(10).to_string(index=False))

    try:
        vacuum_df = databricks_execute(conn, "VACUUM citi_delta.format_comparison RETAIN 168 HOURS DRY RUN", fetch=True)
        print("\\nVACUUM DRY RUN output:")
        print(vacuum_df.head(20).to_string(index=False))
    except Exception as exc:
        print("\\nVACUUM command did not return a result set or was restricted:", exc)

    print("\\n_delta_log mental model:")
    print("- Delta writes JSON commit files plus periodic Parquet checkpoints")
    print("- DESCRIBE HISTORY is the easiest warehouse-safe way to inspect transaction lineage")
    conn.close()


In [ ]:

comparison_df = pd.DataFrame(
    [
        {
            "Capability": "Transaction log location",
            "Delta Lake": "_delta_log inside table path",
            "Apache Iceberg": "Metadata + manifest files referenced from catalog/table metadata",
            "Apache Hudi": ".hoodie timeline inside table path",
        },
        {
            "Capability": "Catalog dependency",
            "Delta Lake": "Optional but strongest inside Databricks metastore / Unity Catalog",
            "Apache Iceberg": "Strong catalog story: Hive, Glue, REST, Nessie",
            "Apache Hudi": "Less catalog-centric; often managed with Hive sync + engines",
        },
        {
            "Capability": "Time travel",
            "Delta Lake": "Yes, by version or timestamp",
            "Apache Iceberg": "Yes, by snapshot",
            "Apache Hudi": "Yes, through commit timeline / incremental semantics",
        },
        {
            "Capability": "Schema evolution",
            "Delta Lake": "Strong and ergonomic in Spark/Databricks",
            "Apache Iceberg": "Strong, ID-based schema evolution",
            "Apache Hudi": "Supported, but ergonomics vary by engine and table type",
        },
        {
            "Capability": "Row-level deletes / MERGE",
            "Delta Lake": "Excellent MERGE / DELETE support",
            "Apache Iceberg": "Strong, increasingly mature row-level ops",
            "Apache Hudi": "Very strong upsert/delete story",
        },
        {
            "Capability": "Streaming support",
            "Delta Lake": "Excellent with Spark Structured Streaming",
            "Apache Iceberg": "Good and improving across engines",
            "Apache Hudi": "Very strong for ingestion and incremental processing",
        },
        {
            "Capability": "Cloud optimization",
            "Delta Lake": "Excellent on Databricks with OPTIMIZE, ZORDER, caching",
            "Apache Iceberg": "Strong open cloud portability across engines",
            "Apache Hudi": "Strong for write-heavy pipelines and ingestion lakes",
        },
        {
            "Capability": "Primary adopters",
            "Delta Lake": "Databricks-centric teams",
            "Apache Iceberg": "Open lakehouse / multi-engine platforms",
            "Apache Hudi": "CDC, ingestion, near-real-time lake pipelines",
        },
    ]
)

print_header("Architecture Comparison")
print(comparison_df.to_string(index=False))


## Iceberg deep dive

### Core architecture

Apache Iceberg is built around a **snapshot-based metadata model**.

Instead of one append-only transaction log file pattern like Delta, Iceberg tracks table state through a chain of metadata objects:

- **metadata.json** files
- **manifest lists**
- **manifest files**
- references to actual data files

That design gives Iceberg a very clean separation between:

- table metadata
- file layout
- engine access

### Why engineers like Iceberg

1. **Hidden partitioning**  
   Users query the table naturally while Iceberg manages partition transforms behind the scenes.  
   That reduces the classic pain of leaking physical partition columns into every pipeline.

2. **Multi-engine interoperability**  
   Iceberg is the strongest fit when the same table must be read and written by:
   - Spark
   - Flink
   - Trino
   - Presto
   - Snowflake
   - Athena
   - other open engines

3. **Catalog-centered design**  
   Iceberg works best with a real catalog such as:
   - Hive Metastore
   - AWS Glue
   - REST catalog
   - Project Nessie

4. **Strong schema evolution**  
   Iceberg uses stable field IDs, which makes renames and schema changes safer than naive name-based approaches.

### When Iceberg wins

Iceberg tends to win when:

- your architecture is **not Databricks-only**
- you want **open table semantics** across engines
- you need **Spark + Flink** or **Spark + Trino** to share the same lake tables cleanly
- you care about long-term vendor flexibility

### Practical summary

If Delta is the best format for a **Databricks-native operating model**, Iceberg is often the best format for an **open, multi-engine lakehouse operating model**.


## Hudi deep dive

### Core architecture

Apache Hudi is built for **record-level change processing**.

Its design is centered on a **timeline** of actions and table services, with strong support for:

- inserts
- upserts
- deletes
- incremental pulls
- compaction and clustering

### Two table types

#### Copy-on-Write (CoW)
- reads are simpler and faster
- writes rewrite files more often
- good when read performance matters more than write latency

#### Merge-on-Read (MoR)
- writes land fast into log files
- reads may need merge work
- good when ingestion is frequent and freshness matters

### Why engineers choose Hudi

1. **High-frequency upserts**  
   Hudi is extremely good when data arrives as continual changes instead of pure append-only batches.

2. **CDC-friendly design**  
   Hudi fits pipelines where Kafka, Debezium, or upstream OLTP systems emit change events all day.

3. **Record-level indexing**  
   Hudi supports indexing strategies that help it find which file group should receive an update.

4. **Bloom filter index and related strategies**  
   These are valuable when locating existing records during heavy upsert workloads.

### When Hudi wins

Hudi tends to win when:

- your workload is **CDC-heavy**
- your lake is receiving **constant upserts**
- freshness matters more than perfectly simple read paths
- you want **incremental processing semantics** baked into the table system

### Practical summary

If Delta feels strongest for **Databricks analytics**, and Iceberg feels strongest for **open multi-engine interoperability**, Hudi feels strongest for **write-heavy, change-heavy ingestion pipelines**.


In [ ]:

decision_df = pd.DataFrame(
    [
        {
            "Scenario": "Databricks workspace, 10TB telemetry lake, daily OPTIMIZE runs, Spark + SQL team",
            "Recommended Format": "Delta Lake",
            "Why": "Best operational fit for Databricks-native table maintenance, SQL ergonomics, MERGE, VACUUM, and observability.",
        },
        {
            "Scenario": "Open architecture with Spark + Flink + Trino across clouds",
            "Recommended Format": "Apache Iceberg",
            "Why": "Best multi-engine interoperability and clean catalog-first metadata model.",
        },
        {
            "Scenario": "High-frequency CDC ingestion with constant upserts",
            "Recommended Format": "Apache Hudi",
            "Why": "Best fit for write-heavy incremental pipelines and record-level update handling.",
        },
    ]
)

print_header("Decision Matrix")
print(decision_df.to_string(index=False))

print_header("Citi recommendation")
print(textwrap.dedent("""
Recommendation: Delta Lake wins for the stated Citi scenario.

Why Delta wins here:
- The team already has a Databricks workspace
- Daily OPTIMIZE-style maintenance is a native operating pattern
- Spark + SQL users benefit from Delta's smoothest developer ergonomics
- Time travel, MERGE, DELETE, VACUUM, and DESCRIBE HISTORY are straightforward
- Operational friction is lowest when the platform is already Databricks-centered

When I would not pick Delta:
- If the same strategic tables must be shared broadly across Spark, Flink, Trino, Athena, and multiple catalogs
- If avoiding platform coupling is the top architectural goal

Bottom line:
- Databricks-first analytics platform -> Delta
- Open multi-engine lakehouse -> Iceberg
- Upsert-heavy CDC lake ingestion -> Hudi
"""))


## What just happened

- **Delta wins on Databricks.**
- **Iceberg wins on multi-engine open architectures.**
- **Hudi wins on upsert-heavy CDC pipelines.**
- In **2026**, Iceberg is gaining ground outside Databricks because more teams want engine flexibility without losing modern table semantics.

### Final mental shortcut

- Pick **Delta** when your platform center of gravity is Databricks.
- Pick **Iceberg** when your architecture center of gravity is openness and engine interoperability.
- Pick **Hudi** when your write path is the hard part and your lake behaves like a CDC system.
